In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.drought_indices import extract_drought_events, calculate_spei
from src.drought_probability import calculate_return_periods

# 1. Locate Data
ensemble_dir = PROJECT_ROOT / "outputs" / "processed" / "cmip6_ensemble"
parquet_files = list(ensemble_dir.glob("sarawak_cmip6_*.parquet"))
parquet_files = [f for f in parquet_files if "master" not in f.name]

if not parquet_files:
    raise FileNotFoundError("No CMIP6 parquet files found.")

print(f"🔍 Found {len(parquet_files)} CMIP6 extraction files.\n")

# FIX 1: Load station mapping correctly
hist_data_path = PROJECT_ROOT / "outputs" / "processed" / "sarawak_ML_corrected.parquet"
if hist_data_path.exists():
    hist_df = pd.read_parquet(hist_data_path)
    hist_df['station_id'] = hist_df['station_id'].astype(str)
    # Create a clean mapping dictionary
    station_mapping = hist_df.drop_duplicates('station_id').set_index('station_id')['station_name'].to_dict()
    print(f"✅ Loaded station mapping for {len(station_mapping)} stations.")
else:
    station_mapping = {}
    print("⚠️ Historical data not found. Using generic station names.")

all_future_events = []

# 2. Process Each File
for file_path in parquet_files:
    filename = file_path.stem
    parts = filename.replace("sarawak_cmip6_", "").split("_")
    
    if len(parts) >= 3:
        model = parts[0]
        scenario = parts[1]
        timeframe = parts[2]
    else:
        continue
        
    print(f"\n🚀 Processing: {model} | {scenario} | {timeframe}")
    
    cmip6_df = pd.read_parquet(file_path)
    cmip6_df['date'] = pd.to_datetime(cmip6_df['date'])
    cmip6_df['station_id'] = cmip6_df['station_id'].astype(str)
    cmip6_df['timeframe'] = timeframe
    
    # FIX 2: Map station names correctly with proper fallback
    cmip6_df['station_name'] = cmip6_df['station_id'].map(station_mapping)
    # Fill NaN with generic names using apply (not f-string with Series)
    cmip6_df['station_name'] = cmip6_df.apply(
        lambda row: row['station_name'] if pd.notna(row['station_name']) else f"Station_{row['station_id']}",
        axis=1
    )
    
    grouped = cmip6_df.groupby(['station_id', 'station_name'])
    
    # DEBUG: Process only first 3 stations to see what's happening
    for idx, ((stn_id, stn_name), group) in enumerate(grouped):
        #if idx >= 3:  # Remove this limit after debugging
        #    break
            
        group = group.sort_values('date').set_index('date')
        
        if 'P_future' not in group.columns or 'PET_future' not in group.columns:
            continue
            
        # Calculate SPEI-12
        spei_12 = calculate_spei(group['P_future'], group['PET_future'], scale=12)
        group['SPEI_12'] = spei_12
        
        # DEBUG: Check SPEI values
        valid_spei = group['SPEI_12'].dropna()
        print(f"\n   🔍 DEBUG {stn_name} (ID: {stn_id}):")
        print(f"      Total months: {len(group)}")
        print(f"      Valid SPEI months: {len(valid_spei)}")
        
        if len(valid_spei) > 0:
            print(f"      Min SPEI: {valid_spei.min():.2f}")
            print(f"      Max SPEI: {valid_spei.max():.2f}")
            print(f"      Months with SPEI <= -1.0: {(valid_spei <= -1.0).sum()}")
            
            if valid_spei.min() < -1.0:
                # Extract events WITHOUT duration filter first to debug
                events_raw = extract_drought_events(group['SPEI_12'], threshold=-1.0, min_duration=1, inter_event_time=1)
                print(f"      Raw events found (min_duration=1): {len(events_raw)}")
                
                if not events_raw.empty:
                    print(f"      Duration distribution:")
                    print(f"        {events_raw['duration'].value_counts().sort_index().to_dict()}")
                
                # Now extract with min_duration=2
                events = extract_drought_events(group['SPEI_12'], threshold=-1.0, min_duration=2, inter_event_time=1)
                print(f"      Events after duration filter (>=2): {len(events)}")
                
                if not events.empty:
                    events = calculate_return_periods(events, len(group))
                    events['model'] = model
                    events['scenario'] = scenario
                    events['timeframe'] = timeframe
                    events['station_id'] = str(stn_id)
                    events['station_name'] = stn_name
                    all_future_events.append(events)
                    print(f"      ✅ Added {len(events)} events")
                else:
                    print(f"      ⚠️ All events filtered out (duration < 2)")
            else:
                print(f"      ℹ️ No drought conditions (Min SPEI >= -1.0)")
        else:
            print(f"      ❌ All SPEI values are NaN")

# 3. Save to CSV
if all_future_events:
    final_future_df = pd.concat(all_future_events, ignore_index=True)
    output_path = PROJECT_ROOT / "outputs" / "processed" / "future_drought_events_summary.csv"
    final_future_df.to_csv(output_path, index=False)
    print(f"\n✅ SUCCESS! Saved {len(final_future_df)} future drought events to: {output_path}")
else:
    print("\n⚠️ No future drought events found.")

c:\Users\User\miniconda3\envs\hydro-ml\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.17) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


🔍 Found 15 CMIP6 extraction files.

✅ Loaded station mapping for 273 stations.

🚀 Processing: ACCESS-CM2 | historical | historical

   🔍 DEBUG Tebedu (ID: 1003031):
      Total months: 360
      Valid SPEI months: 349
      Min SPEI: -3.72
      Max SPEI: 2.58
      Months with SPEI <= -1.0: 21
      Raw events found (min_duration=1): 9
      Duration distribution:
        {1: 6, 4: 1, 5: 1, 6: 1}
      Events after duration filter (>=2): 3
      ✅ Added 3 events

   🔍 DEBUG Krusen (ID: 1004001):
      Total months: 360
      Valid SPEI months: 349
      Min SPEI: -3.72
      Max SPEI: 2.58
      Months with SPEI <= -1.0: 21
      Raw events found (min_duration=1): 9
      Duration distribution:
        {1: 6, 4: 1, 5: 1, 6: 1}
      Events after duration filter (>=2): 3
      ✅ Added 3 events

   🔍 DEBUG Bukit Matuh (ID: 1005079):
      Total months: 360
      Valid SPEI months: 349
      Min SPEI: -3.72
      Max SPEI: 2.60
      Months with SPEI <= -1.0: 19
      Raw events found (m